# Baselines

Establishes the two linear baselines reported in the paper: **Linear Regression**
(raw features) and **Polynomial Regression, degree 2**. This is the only place in
the repository where these two numbers are computed -- without this notebook they
are not reproducible.

Uses the **identical** train/test split (seed=42, 80/20) and 5-fold KFold
(seed=42, shuffle=True) as `01_random_forest.ipynb`, `02_neural_network.ipynb`
and `03_model_comparison.ipynb`.

This notebook runs **last**: its final section assembles the finished
`metrics.json`, `data/comparison_table.csv` and `data/hyperparameters.csv` from the
sections written by notebooks 01-03 plus the baselines computed here.

## Split structure
```
Full dataset  (206 samples)
  └─ Test set   (42 samples, 20%)  ← held out from training; same partition reported for every model
  └─ Train set  (164 samples, 80%) ← used for training + KFold CV
```


In [1]:
import random
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error

# ── Determinism ────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Data loading and shared split

In [2]:
df = pd.read_csv('../data/raw/hepg2.csv')
print(f'Dataset: {df.shape[0]} samples x {df.shape[1]} columns')

X_raw = df[['% DMSO', 'TREHALOSE']].values
y     = df['VIABILIDADE'].values

# -- Shared 80/20 split (identical to the other notebooks) --
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED
)

# -- Shared KFold (identical to the other notebooks) --
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

print(f'Train: {X_train_raw.shape[0]} | Test: {X_test_raw.shape[0]}')
assert (len(df), len(X_train_raw), len(X_test_raw)) == (206, 164, 42), 'Unexpected split sizes'
print('Split sizes match the expected 206 / 164 / 42.')


Dataset: 206 samples x 6 columns
Train: 164 | Test: 42
Split sizes match the expected 206 / 164 / 42.


## Polynomial feature engineering

Used only by the polynomial baseline below. **Fitted on training data only**
to prevent leakage.

In [3]:
# Polynomial features (degree=2, no bias term because LinearRegression adds it)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_raw)   # fit on train only
X_test_poly  = poly.transform(X_test_raw)

feature_names_poly = poly.get_feature_names_out(['DMSO', 'Trehalose'])
print('Polynomial features:', list(feature_names_poly))

Polynomial features: ['DMSO', 'Trehalose', 'DMSO^2', 'DMSO Trehalose', 'Trehalose^2']


## Baseline 1 — Linear Regression (raw features)

In [4]:
lr = LinearRegression()
lr.fit(X_train_raw, y_train)

y_pred_lr = lr.predict(X_test_raw)
r2_lr_test  = r2_score(y_test, y_pred_lr)
rmse_lr_test = np.sqrt(mean_squared_error(y_test, y_pred_lr))

cv_lr = cross_val_score(lr, X_train_raw, y_train, cv=cv, scoring='r2')
r2_lr_cv = cv_lr.mean()

print(f'Linear Regression (raw): R²(test)={r2_lr_test:.4f} | RMSE(test)={rmse_lr_test:.4f} | R²(CV)={r2_lr_cv:.4f}')

Linear Regression (raw): R²(test)=0.5217 | RMSE(test)=25.1770 | R²(CV)=0.2890


## Baseline 2 — Polynomial Regression degree 2

In [5]:
# Polynomial regression = LinearRegression on PolynomialFeatures
pr = LinearRegression()
pr.fit(X_train_poly, y_train)

y_pred_pr = pr.predict(X_test_poly)
r2_pr_test  = r2_score(y_test, y_pred_pr)
rmse_pr_test = np.sqrt(mean_squared_error(y_test, y_pred_pr))

# CV must use poly-transformed features consistently — build a pipeline
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr',   LinearRegression())
])
cv_pr = cross_val_score(poly_pipeline, X_train_raw, y_train, cv=cv, scoring='r2')
r2_pr_cv = cv_pr.mean()

print(f'Polynomial Regression (deg=2): R²(test)={r2_pr_test:.4f} | RMSE(test)={rmse_pr_test:.4f} | R²(CV)={r2_pr_cv:.4f}')

Polynomial Regression (deg=2): R²(test)=0.5823 | RMSE(test)=23.5275 | R²(CV)=0.3654


## Final assembly: `metrics.json` and the exported tables

This is the last step of the pipeline. It adds the `baselines` section and the paper's
example prediction to `metrics.json`, then regenerates `data/comparison_table.csv` and
`data/hyperparameters.csv` from it, so the three files can never drift apart.

The example prediction is the one quoted in a figure legend of the paper:
**10% DMSO, 0% trehalose**.


In [6]:
import json
import sys
import joblib

sys.path.insert(0, '../src')
import rf_inference
import nn_inference

METRICS_PATH = '../metrics.json'
with open(METRICS_PATH, encoding='utf-8') as f:
    metrics = json.load(f)

# ---- baselines computed above -------------------------------------------------
metrics['baselines'] = {
    'linear_regression': {
        'r2_test': round(float(r2_lr_test), 4),
        'rmse_test': round(float(rmse_lr_test), 4),
        'r2_cv': round(float(r2_lr_cv), 4),
    },
    'polynomial_regression_deg2': {
        'r2_test': round(float(r2_pr_test), 4),
        'rmse_test': round(float(rmse_pr_test), 4),
        'r2_cv': round(float(r2_pr_cv), 4),
    },
}

# ---- example prediction quoted in the paper's figure legend --------------------
EX_DMSO, EX_TREH = 10.0, 0.0

rf_sklearn = joblib.load('../models/random_forest_model.pkl')
rf_pred_sklearn = float(rf_sklearn.predict(
    pd.DataFrame({'% DMSO': [EX_DMSO], 'TREHALOSE': [EX_TREH]})
)[0])
rf_pred_numpy = rf_inference.predict_one(EX_DMSO, EX_TREH)
nn_pred_numpy = nn_inference.predict(EX_DMSO, EX_TREH)

assert rf_pred_sklearn == rf_pred_numpy, 'RF NumPy export disagrees with scikit-learn'

example = metrics.get('example_prediction_10dmso_0trehalose', {})
example.update({
    'dmso': EX_DMSO,
    'trehalose': EX_TREH,
    'random_forest': round(rf_pred_sklearn, 4),
    'random_forest_numpy': round(rf_pred_numpy, 4),
    'neural_network_numpy': round(nn_pred_numpy, 4),
})
metrics['example_prediction_10dmso_0trehalose'] = example

print(f'Example prediction ({EX_DMSO:.0f}% DMSO / {EX_TREH:.0f}% trehalose):')
for k in ('random_forest', 'random_forest_numpy', 'neural_network_pytorch', 'neural_network_numpy'):
    if k in example:
        print(f'  {k}: {example[k]}')

with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print()
print(f'Updated {METRICS_PATH} -> sections: {list(metrics)}')


Example prediction (10% DMSO / 0% trehalose):
  random_forest: 91.2245
  random_forest_numpy: 91.2245
  neural_network_pytorch: 89.6854
  neural_network_numpy: 89.6854

Updated ../metrics.json -> sections: ['generated_at', 'dataset', 'random_forest', 'random_forest_numpy_export_validation', 'neural_network', 'example_prediction_10dmso_0trehalose', 'alternatives', 'baselines']


### Exported tables

`data/comparison_table.csv` and `data/hyperparameters.csv` are derived views of
`metrics.json` -- regenerated here so they cannot drift from the source of truth.


In [7]:
rf_m = metrics['random_forest']
nn_m = metrics['neural_network']
alt = metrics['alternatives']
base = metrics['baselines']

comparison_rows = [
    ('Random Forest', rf_m['r2_test'], rf_m['rmse_test'], rf_m['cv_r2_mean']),
    ('Neural Network (ANN)', nn_m['r2_test'], nn_m['rmse_test'], nn_m['cv_r2_mean']),
    ('XGBoost', alt['xgboost']['r2_test'], alt['xgboost']['rmse_test'], alt['xgboost']['r2_cv']),
    ('SVR (raw)', alt['svr']['r2_test'], alt['svr']['rmse_test'], alt['svr']['r2_cv']),
    ('Polynomial Regression (deg=2)', base['polynomial_regression_deg2']['r2_test'],
     base['polynomial_regression_deg2']['rmse_test'], base['polynomial_regression_deg2']['r2_cv']),
    ('Linear Regression (raw)', base['linear_regression']['r2_test'],
     base['linear_regression']['rmse_test'], base['linear_regression']['r2_cv']),
]

comparison_df = (
    pd.DataFrame(comparison_rows, columns=['model', 'R2_test', 'RMSE_test', 'R2_CV'])
    .sort_values('R2_test', ascending=False)
    .set_index('model')
)
comparison_df.index.name = None
comparison_df.to_csv('../data/comparison_table.csv')
print('Wrote ../data/comparison_table.csv')
print(comparison_df.to_string())

wc = nn_m['winning_config']
bn_note = ('used in the winning configuration' if wc['use_bn']
           else 'not used in the winning configuration')
drop_note = ('selected by the validation-loss search' if wc['dropout'] > 0
             else 'not used in the winning configuration')
wd_note = ('L2 regularization' if wc['weight_decay'] > 0
           else 'no L2 regularization in the winning configuration')

hyper_rows = [
    ('Random Forest', 'n_estimators', rf_m['hyperparameters']['n_estimators'], ''),
    ('Random Forest', 'criterion', 'squared_error', 'sklearn default'),
    ('Random Forest', 'max_features', '1.0', 'sklearn 1.4+ default (was sqrt in older versions)'),
    ('Random Forest', 'random_state', rf_m['hyperparameters']['random_state'], ''),
    ('Random Forest', 'n_jobs', 'None', 'sklearn default; n_jobs=-1 is passed only to learning_curve'),
    ('XGBoost', 'n_estimators', '100', ''),
    ('XGBoost', 'learning_rate', '0.1', ''),
    ('XGBoost', 'max_depth', '6', 'default'),
    ('XGBoost', 'subsample', '1.0', 'default'),
    ('XGBoost', 'colsample_bytree', '1.0', 'default'),
    ('XGBoost', 'random_state', '42', ''),
    ('XGBoost', 'verbosity', '0', ''),
    ('SVR', 'kernel', 'rbf', ''),
    ('SVR', 'C', '100', ''),
    ('SVR', 'gamma', '0.1', ''),
    ('SVR', 'epsilon', '0.1', 'default'),
    ('Linear Regression', 'fit_intercept', 'True', 'default; no other hyperparameters'),
    ('Polynomial Regression', 'degree', '2', ''),
    ('Polynomial Regression', 'include_bias', 'False', ''),
    ('Polynomial Regression', 'interaction_only', 'False', 'default'),
    ('Neural Network (ANN)', 'architecture', '128 -> 64 -> 32 -> 1', 'fully connected; linear (unbounded) output layer'),
    ('Neural Network (ANN)', 'input_features', '2', 'raw: % DMSO, TREHALOSE (no polynomial expansion)'),
    ('Neural Network (ANN)', 'batch_normalization', wc['use_bn'], bn_note),
    ('Neural Network (ANN)', 'activation', 'ReLU', ''),
    ('Neural Network (ANN)', 'dropout', wc['dropout'], drop_note),
    ('Neural Network (ANN)', 'optimizer', 'Adam', ''),
    ('Neural Network (ANN)', 'learning_rate', '0.001', ''),
    ('Neural Network (ANN)', 'weight_decay', wc['weight_decay'], wd_note),
    ('Neural Network (ANN)', 'lr_scheduler', 'ReduceLROnPlateau', ''),
    ('Neural Network (ANN)', 'lr_scheduler_factor', '0.5', ''),
    ('Neural Network (ANN)', 'lr_scheduler_patience', '30', ''),
    ('Neural Network (ANN)', 'early_stopping_patience', wc['patience'],
     'fixed before the search; identical for every config'),
    ('Neural Network (ANN)', 'max_epochs', '2000', ''),
    ('Neural Network (ANN)', 'batch_size', '32', ''),
    ('Neural Network (ANN)', 'loss_function', 'MSELoss', ''),
    ('Neural Network (ANN)', 'device', 'cpu', 'forced CPU for reproducibility'),
    ('Neural Network (ANN)', 'seed', '42', 'random + numpy + torch + DataLoader generator'),
    ('Neural Network (ANN)', 'best_epoch', nn_m['best_epoch'], 'from deterministic training run'),
]

hyper_df = pd.DataFrame(hyper_rows, columns=['model', 'hyperparameter', 'value', 'notes'])
hyper_df.to_csv('../data/hyperparameters.csv', index=False)
print()
print(f'Wrote ../data/hyperparameters.csv ({len(hyper_df)} rows)')
print(hyper_df[hyper_df['model'] == 'Neural Network (ANN)'].to_string(index=False))


Wrote ../data/comparison_table.csv
                               R2_test  RMSE_test   R2_CV
Random Forest                   0.9840     4.6043  0.9598
Neural Network (ANN)            0.9818     4.9177  0.9643
XGBoost                         0.9814     4.9602  0.9627
SVR (raw)                       0.6485    21.5835  0.3460
Polynomial Regression (deg=2)   0.5823    23.5275  0.3654
Linear Regression (raw)         0.5217    25.1770  0.2890

Wrote ../data/hyperparameters.csv (38 rows)
               model          hyperparameter                value                                               notes
Neural Network (ANN)            architecture 128 -> 64 -> 32 -> 1    fully connected; linear (unbounded) output layer
Neural Network (ANN)          input_features                    2    raw: % DMSO, TREHALOSE (no polynomial expansion)
Neural Network (ANN)     batch_normalization                 True                   used in the winning configuration
Neural Network (ANN)              activati